# XAUUSD LLM Forecaster — Colab Pipeline

**Before running:**
1. Runtime → Change runtime type → **GPU** (T4 or better)
2. Upload `XAU_30m_data.csv` to Google Drive at `MyDrive/xauusd-llm-forecaster/data/raw/`
3. Run all cells top-to-bottom

**Workflow:**
- **Code** → pulled from GitHub each session (no uploads needed)
- **Data + Checkpoints** → saved to Drive (persist between sessions)

## 0. Setup — Mount Drive, Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys

REPO_URL  = 'https://github.com/211abhi/xauusd-llm-forecaster.git'
CODE_DIR  = '/content/xauusd-llm-forecaster'
DRIVE_DIR = '/content/drive/MyDrive/xauusd-llm-forecaster'  # data + checkpoints live here

# Pull latest code from GitHub (clone on first run, pull on subsequent runs)
if not os.path.isdir(CODE_DIR):
    !git clone {REPO_URL} {CODE_DIR}
else:
    !git -C {CODE_DIR} pull

# Set working dir and Python path so src.* imports work everywhere
os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)
os.environ['PYTHONPATH'] = CODE_DIR

# Create required dirs on Drive (persistent across sessions)
for d in ['data/raw', 'data/processed', 'data/splits',
          'checkpoints/encoder', 'checkpoints/soft_prompt', 'checkpoints/pred_head',
          'logs']:
    os.makedirs(os.path.join(DRIVE_DIR, d), exist_ok=True)

print(f'Code       : {CODE_DIR}  (GitHub)')
print(f'Persistent : {DRIVE_DIR}  (Drive)')

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers>=4.40.0 datasets ta cma scikit-learn pyyaml tqdm matplotlib seaborn

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
import yaml

cfg_path = os.path.join(CODE_DIR, 'configs/base_config.yaml')
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

# Point all I/O paths to Drive so data and checkpoints persist between sessions
cfg['project']['device']              = device
cfg['data']['raw_path']               = f'{DRIVE_DIR}/data/raw/XAU_30m_data.csv'
cfg['data']['processed_dir']          = f'{DRIVE_DIR}/data/processed/'
cfg['data']['splits_dir']             = f'{DRIVE_DIR}/data/splits/'
cfg['project']['checkpoint_dir']      = f'{DRIVE_DIR}/checkpoints/'
cfg['project']['log_dir']             = f'{DRIVE_DIR}/logs/'
cfg['encoder']['checkpoint_path']     = f'{DRIVE_DIR}/checkpoints/encoder/best_encoder.pt'
cfg['soft_prompt']['checkpoint_path'] = f'{DRIVE_DIR}/checkpoints/soft_prompt/best_soft_prompt.npy'
cfg['prediction_head']['checkpoint_path'] = f'{DRIVE_DIR}/checkpoints/pred_head/best_pred_head.pt'
cfg['llm']['cache_dir']               = f'{DRIVE_DIR}/.cache/llm'

with open(cfg_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('Config updated. Paths:')
print(f"  raw data    : {cfg['data']['raw_path']}")
print(f"  processed   : {cfg['data']['processed_dir']}")
print(f"  checkpoints : {cfg['project']['checkpoint_dir']}")
print(f"  device      : {cfg['project']['device']}")

## Phase 1 — Data Preparation

Cleans raw OHLCV, adds indicators, normalizes, and saves train/val/test splits.

In [ ]:
!python scripts/01_prepare_data.py --config configs/base_config.yaml

In [ ]:
# Quick sanity check
import pandas as pd
df = pd.read_csv('data/processed/xau_processed.csv', parse_dates=['datetime'])
print(f'Processed rows: {len(df)}')
print(df.describe().round(3))

## Phase 1b — Regime Labeling

Assigns rule-based market regime labels (TRENDING_UP / DOWN / RANGING / VOLATILE / BREAKOUT).

In [ ]:
!python scripts/02_label_regimes.py --config configs/base_config.yaml

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv('data/processed/xau_with_regimes.csv', parse_dates=['datetime'])
counts = df['regime'].value_counts()
counts.plot(kind='bar', figsize=(8, 4), title='Regime Distribution (per candle)', color='steelblue')
plt.tight_layout()
plt.show()
print(counts)

## Phase 2 — Train TS Encoder (Contrastive)

Trains the 1D-Conv + Transformer encoder against frozen LLM text embeddings via InfoNCE loss.

⏱ ~20–40 min on T4 depending on dataset size.

In [ ]:
!python scripts/03_train_encoder.py --config configs/base_config.yaml

In [ ]:
from pathlib import Path
ckpt = Path('checkpoints/encoder/best_encoder.pt')
print('Encoder checkpoint saved:', ckpt.exists())
print(f'Size: {ckpt.stat().st_size / 1e6:.1f} MB')

## Phase 3 — CMA-ES Soft Prompt Optimization

Optimizes the (32, 768) soft prompt prefix in a 500-dim subspace using evolutionary search.

⏱ ~1–3 hours on T4 (300 generations × 20 candidates). Reduce `maxiter` in config to speed up.

**To speed up:** Edit `configs/base_config.yaml` → `cmaes.maxiter: 50` for a quick test run.

In [ ]:
# Optional: reduce iterations for a quick test
# import yaml
# with open('configs/base_config.yaml') as f: cfg = yaml.safe_load(f)
# cfg['cmaes']['maxiter'] = 50
# with open('configs/base_config.yaml', 'w') as f: yaml.dump(cfg, f)

!python scripts/04_optimize_soft_prompt.py --config configs/base_config.yaml

In [ ]:
import numpy as np
from pathlib import Path
sp_path = Path('checkpoints/soft_prompt/best_soft_prompt.npy')
print('Soft prompt saved:', sp_path.exists())
if sp_path.exists():
    arr = np.load(str(sp_path))
    print(f'Shape: {arr.shape}  mean={arr.mean():.4f}  std={arr.std():.4f}')

## Phase 4 — Train Prediction Head

Trains only the 2-layer MLP on frozen encoder + frozen LLM + frozen soft prompt.

⏱ ~5–15 min on T4.

In [ ]:
!python scripts/05_train_pred_head.py --config configs/base_config.yaml

In [ ]:
from pathlib import Path
ph_path = Path('checkpoints/pred_head/best_pred_head.pt')
print('Pred head saved:', ph_path.exists())

## Phase 5 — Evaluation

Runs the full pipeline on the held-out test set and reports MAE, RMSE, and directional accuracy.

In [ ]:
!python scripts/06_evaluate.py --config configs/base_config.yaml

In [ ]:
import json
with open('logs/evaluation_results.json') as f:
    results = json.load(f)

print('=== Overall ===')
for k, v in results['overall'].items():
    print(f'  {k}: {v:.4f}')

if 'per_regime' in results:
    print('\n=== Per Regime ===')
    for regime, m in sorted(results['per_regime'].items()):
        print(f"  {regime:15s}  MAE={m['mae']:.4f}  DirAcc={m['directional_accuracy']:.1f}%")

## Quick Inference — Single Prediction

Run this after all phases are complete to test the forecaster on a single window.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, yaml
from src.prediction.forecaster import Forecaster
from src.utils.data_loader import get_feature_columns

with open('configs/base_config.yaml') as f:
    cfg = yaml.safe_load(f)

forecaster = Forecaster.from_config(cfg)

df = pd.read_csv('data/processed/xau_with_regimes.csv', parse_dates=['datetime'])
feature_cols = get_feature_columns(cfg)
arr = df[feature_cols].values

W = cfg['tokenizer']['window_size']
# Take the last window before the test set
window = arr[-W - 8: -8][np.newaxis]   # (1, W, F)
actual = arr[-8:, feature_cols.index('close')]  # 8 actual future closes

preds = forecaster.predict(window)[0]  # (8,)

plt.figure(figsize=(10, 4))
plt.plot(range(8), actual, 'o-', label='Actual', color='steelblue')
plt.plot(range(8), preds,  's--', label='Predicted', color='tomato')
plt.title('8-Step Ahead Forecast (normalized close price)')
plt.xlabel('Step (×30 min)')
plt.legend()
plt.tight_layout()
plt.show()

mae = abs(preds - actual).mean()
print(f'MAE on this window: {mae:.4f}')